# Backtester Core Engine

**Owner:** James  
**Version:** 1.0 (Feb 6, 2026)  

Core backtesting engine for evaluating sports betting strategies.  
See `docs/reference/strategy-interface.md` for the strategy function contract.

## Output Format

| Column | Type | Description |
|--------|------|-------------|
| timestamp | datetime | When trade happened |
| game | str | "Home vs Away" |
| action | str | 'BUY_HOME' or 'BUY_AWAY' |
| bet_size | float | Dollars bet |
| odds | float | Decimal odds used |
| outcome | str | 'WIN' or 'LOSS' |
| pnl | float | Profit/loss for this trade |
| cumulative_pnl | float | Running total P&L |
| bankroll | float | Current bankroll after trade |

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Callable

import pandas as pd

## Configuration

In [2]:
# Project root (one level up from tools/)
PROJECT_ROOT = Path("__file__").resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Default CSV paths
DUMMY_CSV = DATA_DIR / "dummy_backtest_input.csv"
TEST_CSV = DATA_DIR / "test_games.csv"

## `load_backtest_data()`

Loads game data for backtesting. Tries Railway database first, falls back to local CSV.

In [3]:
def load_backtest_data(
    start_date: str,
    end_date: str,
    csv_path: str | Path | None = None,
) -> pd.DataFrame:
    """Load game data for backtesting.

    Attempts to load from Railway PostgreSQL database first. If DATABASE_URL
    is not set or the connection fails, falls back to loading from a local CSV.

    Args:
        start_date: Start date filter in "YYYY-MM-DD" format.
        end_date: End date filter in "YYYY-MM-DD" format.
        csv_path: Path to fallback CSV file. Defaults to dummy_backtest_input.csv.

    Returns:
        DataFrame with columns: timestamp, game, home_team, away_team,
        home_odds, away_odds, home_win. Sorted by timestamp ascending.

    Raises:
        FileNotFoundError: If no database and no CSV file found.

    Example:
        >>> data = load_backtest_data("2026-01-01", "2026-01-31")
        >>> print(data.columns.tolist())
        ['timestamp', 'game', 'home_team', 'away_team', 'home_odds', 'away_odds', 'home_win']
    """
    database_url = os.environ.get("DATABASE_URL")

    if database_url:
        try:
            from sqlalchemy import create_engine, text

            engine = create_engine(database_url)
            query = text("""
                SELECT
                    m.commence_time AS timestamp,
                    m.home_team || ' vs ' || m.away_team AS game,
                    m.home_team,
                    m.away_team,
                    o.home_odds,
                    o.away_odds,
                    m.home_win
                FROM sportsbook_matches m
                JOIN sportsbook_odds o ON m.id = o.match_id
                WHERE m.commence_time >= :start_date
                  AND m.commence_time <= :end_date
                ORDER BY m.commence_time ASC
            """)

            with engine.connect() as conn:
                df = pd.read_sql(query, conn, params={"start_date": start_date, "end_date": end_date})

            df["timestamp"] = pd.to_datetime(df["timestamp"])
            print(f"Loaded {len(df)} rows from Railway database.")
            return df

        except Exception as e:
            print(f"Database connection failed ({e}), falling back to CSV.")

    # Fallback to CSV
    if csv_path is None:
        csv_path = DUMMY_CSV
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"No CSV found at {csv_path}")

    df = pd.read_csv(csv_path, parse_dates=["timestamp"])

    # Filter by date range
    mask = (df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)
    df = df.loc[mask].sort_values("timestamp").reset_index(drop=True)

    print(f"Loaded {len(df)} rows from {csv_path.name}.")
    return df

## `backtest()`

Core backtesting loop. Takes game data and a strategy function, simulates betting, and returns a trade log.

In [ ]:
def backtest(
    data: pd.DataFrame,
    strategy_fn: Callable[[pd.Series, dict[str, Any] | None], dict[str, Any]],
    initial_bankroll: float = 10000.0,
) -> pd.DataFrame:
    """Run a backtest over historical game data using a strategy function.

    Iterates through each game, calls the strategy function to get a signal,
    and simulates the bet outcome. Tracks bankroll, P&L, and trade history.

    Args:
        data: DataFrame from load_backtest_data() with columns: timestamp,
            game, home_team, away_team, home_odds, away_odds, home_win.
        strategy_fn: Callable matching the strategy interface. Takes
            (row: pd.Series, context: dict | None) and returns a dict with
            keys: action, confidence, size, reason (optional).
        initial_bankroll: Starting bankroll in dollars. Defaults to 10000.

    Returns:
        DataFrame with 9 columns: timestamp, game, action, bet_size, odds,
        outcome, pnl, cumulative_pnl, bankroll. Returns empty DataFrame with
        correct columns if no trades are executed.

    Example:
        >>> data = load_backtest_data("2026-01-01", "2026-01-31")
        >>> results = backtest(data, always_bet_home)
        >>> print(results[["game", "action", "pnl", "bankroll"]])
    """
    # Define output columns upfront (fixes empty DataFrame having no columns)
    OUTPUT_COLUMNS = [
        "timestamp", "game", "action", "bet_size",
        "odds", "outcome", "pnl", "cumulative_pnl", "bankroll",
    ]

    bankroll = initial_bankroll
    cumulative_pnl = 0.0
    trades: list[dict[str, Any]] = []

    context: dict[str, Any] = {
        "initial_bankroll": initial_bankroll,
        "bankroll": bankroll,
        "trade_count": 0,
        "cumulative_pnl": 0.0,
    }

    for _, row in data.iterrows():
        if bankroll <= 0:
            break

        # Skip rows with NaN odds (prevents corruption of subsequent rows)
        if pd.isna(row["home_odds"]) or pd.isna(row["away_odds"]):
            continue

        # Update context for strategy
        context["bankroll"] = bankroll
        context["trade_count"] = len(trades)
        context["cumulative_pnl"] = cumulative_pnl

        # Remove outcome column to prevent data leakage
        strategy_row = row.drop(labels=["home_win"])
        signal = strategy_fn(strategy_row, context)
        action = signal.get("action", "SKIP")

        if action == "SKIP":
            continue

        # Determine bet size (cap at current bankroll)
        bet_size = min(signal.get("size", 0.0), bankroll)
        if bet_size <= 0:
            continue

        # Determine odds based on action
        if action == "BUY_HOME":
            odds = row["home_odds"]
            won = row["home_win"] == 1
        elif action == "BUY_AWAY":
            odds = row["away_odds"]
            won = row["home_win"] == 0
        else:
            continue  # Invalid action, skip

        # Calculate P&L
        if won:
            pnl = bet_size * (odds - 1)
            outcome = "WIN"
        else:
            pnl = -bet_size
            outcome = "LOSS"

        cumulative_pnl += pnl
        bankroll += pnl

        trades.append({
            "timestamp": row["timestamp"],
            "game": row["game"],
            "action": action,
            "bet_size": round(bet_size, 2),
            "odds": odds,
            "outcome": outcome,
            "pnl": round(pnl, 2),
            "cumulative_pnl": round(cumulative_pnl, 2),
            "bankroll": round(bankroll, 2),
        })

    # Return DataFrame with correct columns even if empty
    if not trades:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    return pd.DataFrame(trades)[OUTPUT_COLUMNS]

## Example Strategy: Always Bet Home

A simple strategy that always bets on the home team with a fixed stake. Used for testing the backtester.

In [5]:
def always_bet_home(
    row: pd.Series,
    context: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Example strategy: always bet on the home team.

    Bets a fixed $100 on the home team for every game.
    This is a naive strategy used to validate the backtester.

    Args:
        row: Game data row with home_odds, away_odds, etc.
        context: Optional backtester context (bankroll, trade count, etc.).

    Returns:
        Signal dict with action='BUY_HOME', confidence=0.5, size=100.
    """
    return {
        "action": "BUY_HOME",
        "confidence": 0.5,
        "size": 100.0,
        "reason": "Always bet home (test strategy)",
    }

## End-to-End Test Run

Load dummy data and run the backtester with the "always bet home" strategy.

In [6]:
# Load dummy data
data = load_backtest_data("2026-01-01", "2026-01-31")
print(f"Games loaded: {len(data)}")
data.head()

Loaded 25 rows from dummy_backtest_input.csv.
Games loaded: 25


,timestamp,game,home_team,away_team,home_odds,away_odds,home_win
0,2026-01-01,Lakers vs Celtics,Lakers,Celtics,1.95,2.05,1
1,2026-01-02,Warriors vs Heat,Warriors,Heat,1.80,2.20,1
2,2026-01-03,Bucks vs Nets,Bucks,Nets,1.65,2.45,1
3,2026-01-04,Nuggets vs Suns,Nuggets,Suns,2.10,1.85,0
4,2026-01-05,76ers vs Knicks,76ers,Knicks,1.90,2.00,1


In [7]:
# Run backtest
results = backtest(data, always_bet_home, initial_bankroll=10000.0)
print(f"Total trades: {len(results)}")
print(f"Columns: {results.columns.tolist()}")
results

Total trades: 25
Columns: ['timestamp', 'game', 'action', 'bet_size', 'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll']


,timestamp,game,action,bet_size,odds,outcome,pnl,cumulative_pnl,bankroll
0,2026-01-01,Lakers vs Celtics,BUY_HOME,100.0,1.95,WIN,95.0,95.0,10095.0
1,2026-01-02,Warriors vs Heat,BUY_HOME,100.0,1.80,WIN,80.0,175.0,10175.0
2,2026-01-03,Bucks vs Nets,BUY_HOME,100.0,1.65,WIN,65.0,240.0,10240.0
3,2026-01-04,Nuggets vs Suns,BUY_HOME,100.0,2.10,LOSS,-100.0,140.0,10140.0
4,2026-01-05,76ers vs Knicks,BUY_HOME,100.0,1.90,WIN,90.0,230.0,10230.0
5,2026-01-06,Celtics vs Heat,BUY_HOME,100.0,1.70,WIN,70.0,300.0,10300.0
6,2026-01-07,Lakers vs Warriors,BUY_HOME,100.0,2.15,LOSS,-100.0,200.0,10200.0
7,2026-01-08,Nets vs Bucks,BUY_HOME,100.0,2.40,LOSS,-100.0,100.0,10100.0
8,2026-01-09,Suns vs 76ers,BUY_HOME,100.0,1.95,WIN,95.0,195.0,10195.0
9,2026-01-10,Knicks vs Nuggets,BUY_HOME,100.0,2.25,LOSS,-100.0,95.0,10095.0


In [8]:
# Summary statistics
if len(results) > 0:
    wins = (results["outcome"] == "WIN").sum()
    losses = (results["outcome"] == "LOSS").sum()
    win_rate = wins / len(results)
    final_pnl = results["cumulative_pnl"].iloc[-1]
    final_bankroll = results["bankroll"].iloc[-1]

    print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
    print(f"Total P&L:       ${final_pnl:,.2f}")
    print(f"Final Bankroll:  ${final_bankroll:,.2f}")
    print(f"ROI:             {final_pnl / 10000:.1%}")
else:
    print("No trades executed.")

Win Rate:        60.0% (15W / 10L)
Total P&L:       $305.00
Final Bankroll:  $10,305.00
ROI:             3.0%


## Verify Output Format

Confirm the results DataFrame has exactly the 9 required columns with correct types.

In [9]:
# Validate output schema
REQUIRED_COLUMNS = [
    "timestamp", "game", "action", "bet_size",
    "odds", "outcome", "pnl", "cumulative_pnl", "bankroll",
]

assert results.columns.tolist() == REQUIRED_COLUMNS, (
    f"Column mismatch!\n"
    f"Expected: {REQUIRED_COLUMNS}\n"
    f"Got:      {results.columns.tolist()}"
)

assert len(results) > 0, "No trades were produced"
assert all(results["action"].isin(["BUY_HOME", "BUY_AWAY"])), "Invalid action values"
assert all(results["outcome"].isin(["WIN", "LOSS"])), "Invalid outcome values"
assert all(results["bet_size"] > 0), "Bet sizes must be positive"

print("All output format checks passed!")

All output format checks passed!


## Test with Mya's `test_games.csv`

Run the backtester against Mya's 100-row test dataset to validate compatibility.

In [10]:
# Load Mya's test data (100 rows)
mya_data = load_backtest_data("2026-01-01", "2026-12-31", csv_path=TEST_CSV)
print(f"Games loaded: {len(mya_data)}")
print(f"Columns: {mya_data.columns.tolist()}")
mya_data.head()

Loaded 100 rows from test_games.csv.
Games loaded: 100
Columns: ['timestamp', 'game', 'home_team', 'away_team', 'home_odds', 'away_odds', 'home_win']


,timestamp,game,home_team,away_team,home_odds,away_odds,home_win
0,2026-01-01 00:00:00,Brooklyn Nets vs Phoenix Suns,Brooklyn Nets,Phoenix Suns,2.388,2.267,0
1,2026-01-01 00:01:00,LA Clippers vs Miami Heat,LA Clippers,Miami Heat,2.511,2.365,0
2,2026-01-01 00:02:00,Portland Trail Blazers vs Los Angeles Lakers,Portland Trail Blazers,Los Angeles Lakers,2.875,2.208,0
3,2026-01-01 00:03:00,Milwaukee Bucks vs Houston Rockets,Milwaukee Bucks,Houston Rockets,2.110,2.208,1
4,2026-01-01 00:04:00,Washington Wizards vs Portland Trail Blazers,Washington Wizards,Portland Trail Blazers,2.668,1.890,0


In [11]:
# Run backtest on Mya's data
mya_results = backtest(mya_data, always_bet_home, initial_bankroll=10000.0)
print(f"Total trades: {len(mya_results)}")
print(f"Columns: {mya_results.columns.tolist()}")
mya_results.head(10)

Total trades: 100
Columns: ['timestamp', 'game', 'action', 'bet_size', 'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll']


,timestamp,game,action,bet_size,odds,outcome,pnl,cumulative_pnl,bankroll
0,2026-01-01 00:00:00,Brooklyn Nets vs Phoenix Suns,BUY_HOME,100.0,2.388,LOSS,-100.0,-100.0,9900.0
1,2026-01-01 00:01:00,LA Clippers vs Miami Heat,BUY_HOME,100.0,2.511,LOSS,-100.0,-200.0,9800.0
2,2026-01-01 00:02:00,Portland Trail Blazers vs Los Angeles Lakers,BUY_HOME,100.0,2.875,LOSS,-100.0,-300.0,9700.0
3,2026-01-01 00:03:00,Milwaukee Bucks vs Houston Rockets,BUY_HOME,100.0,2.110,WIN,111.0,-189.0,9811.0
4,2026-01-01 00:04:00,Washington Wizards vs Portland Trail Blazers,BUY_HOME,100.0,2.668,LOSS,-100.0,-289.0,9711.0
5,2026-01-01 00:05:00,New Orleans Pelicans vs Los Angeles Lakers,BUY_HOME,100.0,2.114,LOSS,-100.0,-389.0,9611.0
6,2026-01-01 00:06:00,Memphis Grizzlies vs Boston Celtics,BUY_HOME,100.0,3.000,LOSS,-100.0,-489.0,9511.0
7,2026-01-01 00:07:00,LA Clippers vs Chicago Bulls,BUY_HOME,100.0,2.017,WIN,101.7,-387.3,9612.7
8,2026-01-01 00:08:00,Los Angeles Lakers vs Charlotte Hornets,BUY_HOME,100.0,2.237,WIN,123.7,-263.6,9736.4
9,2026-01-01 00:09:00,Los Angeles Lakers vs Chicago Bulls,BUY_HOME,100.0,2.518,LOSS,-100.0,-363.6,9636.4


In [ ]:
# Summary statistics for Mya's data
if len(mya_results) > 0:
    wins = (mya_results["outcome"] == "WIN").sum()
    losses = (mya_results["outcome"] == "LOSS").sum()
    win_rate = wins / len(mya_results)
    final_pnl = mya_results["cumulative_pnl"].iloc[-1]
    final_bankroll = mya_results["bankroll"].iloc[-1]

    print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
    print(f"Total P&L:       ${final_pnl:,.2f}")
    print(f"Final Bankroll:  ${final_bankroll:,.2f}")
    print(f"ROI:             {final_pnl / 10000:.1%}")
else:
    print("No trades executed.")

In [ ]:
# Validate 9-column output format for Mya's data
assert mya_results.columns.tolist() == REQUIRED_COLUMNS, (
    f"Column mismatch!\n"
    f"Expected: {REQUIRED_COLUMNS}\n"
    f"Got:      {mya_results.columns.tolist()}"
)

assert len(mya_results) > 0, "No trades were produced"
assert all(mya_results["action"].isin(["BUY_HOME", "BUY_AWAY"])), "Invalid action values"
assert all(mya_results["outcome"].isin(["WIN", "LOSS"])), "Invalid outcome values"
assert all(mya_results["bet_size"] > 0), "Bet sizes must be positive"

print("All output format checks passed for Mya's test_games.csv!")

## Edge Case Tests

Tests for feedback items:
1. Empty results should return DataFrame with correct 9 columns
2. NaN odds should be skipped without corrupting subsequent rows

In [ ]:
# Test 1: Skip-all strategy should return empty DataFrame with correct columns
def skip_all_strategy(row, context=None):
    """Strategy that skips every game."""
    return {"action": "SKIP", "confidence": 0, "size": 0}

empty_results = backtest(data, skip_all_strategy)
print(f"Empty results shape: {empty_results.shape}")
print(f"Empty results columns: {empty_results.columns.tolist()}")

assert empty_results.columns.tolist() == REQUIRED_COLUMNS, (
    f"Empty DataFrame should have correct columns!
"
    f"Expected: {REQUIRED_COLUMNS}
"
    f"Got: {empty_results.columns.tolist()}"
)
assert len(empty_results) == 0, "Should have no trades"
print("Test 1 PASSED: Empty results have correct 9 columns!")

In [ ]:
# Test 2: NaN odds should be skipped without errors
import numpy as np

# Create test data with some NaN odds
nan_test_data = data.copy()
nan_test_data.loc[2, "home_odds"] = np.nan  # Row 3 has NaN home_odds
nan_test_data.loc[5, "away_odds"] = np.nan  # Row 6 has NaN away_odds
print(f"Test data rows: {len(nan_test_data)}")
print(f"Rows with NaN odds: 2 (indices 2 and 5)")

# Run backtest - should skip NaN rows and continue
nan_results = backtest(nan_test_data, always_bet_home)
print(f"Results rows: {len(nan_results)} (expected: {len(data) - 2})")

# Verify we got 2 fewer trades (the NaN rows were skipped)
assert len(nan_results) == len(data) - 2, (
    f"Should skip NaN rows! Expected {len(data) - 2} trades, got {len(nan_results)}"
)

# Verify no NaN values in results
assert not nan_results["odds"].isna().any(), "Results should have no NaN odds"
print("Test 2 PASSED: NaN odds rows are skipped correctly!")

In [ ]:
print("
" + "="*50)
print("ALL EDGE CASE TESTS PASSED!")
print("="*50)